# SynthTextEval evaluation pipeline for ICL-PILOT

This notebook adapts the [SynthTextEval](https://github.com/kr-ramesh/synthtexteval) pipeline to the traceable ENNI-style narratives in ICL-PILOT. It evaluates generated versus real text with corpus-level descriptive and automated text-quality measures, downstream TD/SLI classification, age-cohort performance/fairness auditing, and reference-overlap privacy diagnostics.

The downstream-classification analysis uses rotating pair-level splits. **Predictions are evaluated within each training round and only then summarized across rounds**; repeated predictions of the same underlying child across different trained models are not treated as independent test observations. Split membership uses the composite `(cohort_year, pair_id)` key throughout, so reused `pair_id` values across cohorts cannot contaminate a split.

The balanced descriptive/text-quality comparison controls cohort and TD/SLI composition, but the selected synthetic and real rows are distributional samples rather than semantic pairs. The privacy section instead uses the full available synthetic corpus and full available real reference corpus, because a balanced subsample would undercount possible entity/context overlap.

The package's entity-leakage and entity-phrase-overlap analyses remain **reference-overlap diagnostics**, not verified privacy-leakage estimates: no supplied list of protected identifiers exists for these de-identified stories. Canary attacks are not run because the generator was not trained with planted canaries. The coreference module remains out of scope because the export has no gold coreference annotation.

At the end, the notebook writes a reproducibility manifest, copies a compact result bundle to Google Drive, creates and verifies a ZIP, and starts a local download. Runtime unassignment is deliberately placed in the final cell and requires explicit confirmation that the browser download has completed.


In [ ]:
import warnings

# Suppress the noisy Python-3.13 warning emitted by Jupyter itself without hiding
# deprecations raised by SynthTextEval, Transformers, or our own code.
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module=r"jupyter_client(?:\..*)?$",
)


In [ ]:
# Repository paths and run configuration.
from pathlib import Path
import sys

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
COLAB_REPO_DIR = Path("/content/ICL-PILOT")
candidate_roots = [COLAB_REPO_DIR, Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in candidate_roots if (path / "pyproject.toml").exists()), None)

if PROJECT_ROOT is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Run this notebook from the ICL-PILOT repository or open it in Colab.")
    !git clone {REPO_URL} {COLAB_REPO_DIR}
    PROJECT_ROOT = COLAB_REPO_DIR

%cd {PROJECT_ROOT}

TEXT_RECORDS_CSV = PROJECT_ROOT / "phase2/generated/all_ages/text/all_story_text_records_4_to_7.csv"
GILLAM_EXTERNAL_TEXT_CSV = PROJECT_ROOT / "phase2/Gillam/text/gillam_external_text_records.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs/synthtexteval_demo"
TEXT_COLUMN = "story_text"
SOURCE_COLUMN = "source"
GROUP_COLUMN = "group"
COHORT_COLUMN = "cohort_year"
COMPARISON_COHORTS = (4, 5, 6, 7)
COMPARISON_GROUPS = ("TD", "SLI")
SAMPLE_SEED = 42
MAX_SAMPLES_PER_GROUP = None  # Set to 20 for a CPU smoke test.

RUN_MAUVE = True
RUN_REPEATED_MAUVE = True
REPEATED_MAUVE_N_RUNS = 20
MAUVE_STABILITY_BASE_SEED = 42
MAUVE_DEVICE_ID = 0
MAUVE_MAX_TEXT_LENGTH = 512
# In addition to varying MAUVE's seed on the full balanced corpus, run an 80%
# subsampling sensitivity analysis. Set to None to disable the subsampling arm.
MAUVE_SUBSAMPLE_FRACTION = 0.80

RUN_CLASSIFICATION = True  # Requires a CUDA-enabled Colab runtime.
RUN_GILLAM_EXTERNAL_TEST = True  # Gillam remains test-only.
RUN_PRIVACY_ANALYSIS = True
PRIVACY_ENTITY_LABELS = ("PERSON", "ORG", "GPE", "LOC", "FAC", "NORP")
PRIVACY_MAX_CONTEXT_WINDOW = 4
CLASSIFICATION_ROUNDS = ("round_01", "round_02", "round_03", "round_04")
CLASSIFIER_MODEL = "bert-base-uncased"
LABEL_MAPPING = {"SLI": 0, "TD": 1}
META_RESPONSE_MARKERS = ("I replaced the main characters to match the target story",)

# Export settings. Model checkpoints are large and are excluded from the portable
# ZIP by default; predictions, metrics, split definitions, manifests, and plots are kept.
DRIVE_EXPORT_ROOT = Path("/content/drive/MyDrive/ICL-PILOT/synthtexteval_exports")
EXPORT_INCLUDE_MODEL_CHECKPOINTS = False
EXPORT_ARCHIVE_PREFIX = "synthtexteval_demo"

assert TEXT_RECORDS_CSV.exists(), f"Missing text record: {TEXT_RECORDS_CSV}"
assert GILLAM_EXTERNAL_TEXT_CSV.exists(), f"Missing Gillam external-test export: {GILLAM_EXTERNAL_TEXT_CSV}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print({
    "project_root": str(PROJECT_ROOT),
    "text_records": str(TEXT_RECORDS_CSV),
    "gillam_external_records": str(GILLAM_EXTERNAL_TEXT_CSV),
    "results_dir": str(RESULTS_DIR),
})


In [ ]:
# Install SynthTextEval from a local clone and apply narrowly scoped compatibility fixes.
# A sentinel prevents a reinstall loop after the one required runtime restart.
from pathlib import Path
import re
import subprocess
import sys

SYNTHTEXTEVAL_REVISION = "3d63a172418a985fd0d997e5c993f6a0390c294e"
SYNTHTEXTEVAL_DIR = Path("/content/synthtexteval-src")
INSTALL_SENTINEL = SYNTHTEXTEVAL_DIR / ".icl_pilot_install_ready"


def run_checked(command):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(part) for part in command], check=True)


if not (SYNTHTEXTEVAL_DIR / ".git").exists():
    run_checked(["git", "clone", "https://github.com/kr-ramesh/synthtexteval.git", SYNTHTEXTEVAL_DIR])

sentinel_matches = (
    INSTALL_SENTINEL.exists()
    and INSTALL_SENTINEL.read_text(encoding="utf-8").strip() == SYNTHTEXTEVAL_REVISION
)

if not sentinel_matches:
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "fetch", "--all", "--tags", "-q"])
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "checkout", "-q", SYNTHTEXTEVAL_REVISION])
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "reset", "--hard", "-q", SYNTHTEXTEVAL_REVISION])

    # Packaging fix: this repository revision lacks synthtexteval/eval/__init__.py,
    # while setup.py relies on package discovery.
    eval_init = SYNTHTEXTEVAL_DIR / "synthtexteval" / "eval" / "__init__.py"
    eval_init.touch(exist_ok=True)

    # Transformers compatibility fix. Keep tokenization semantics unchanged:
    # encode_plus -> tokenizer(...) and pad_to_max_length=True -> padding="max_length".
    classifier_utils_path = (
        SYNTHTEXTEVAL_DIR
        / "synthtexteval"
        / "eval"
        / "downstream"
        / "classify"
        / "utils.py"
    )
    utils_text = classifier_utils_path.read_text(encoding="utf-8")
    utils_text, encode_plus_count = re.subn(
        r"tokenizer\s*\.\s*encode_plus\s*\(",
        "tokenizer(",
        utils_text,
    )
    utils_text, legacy_padding_count = re.subn(
        r"pad_to_max_length\s*=\s*True",
        'padding="max_length"',
        utils_text,
    )
    classifier_utils_path.write_text(utils_text, encoding="utf-8")

    assert re.search(r"tokenizer\s*\.\s*encode_plus\s*\(", utils_text) is None
    assert "pad_to_max_length" not in utils_text
    assert 'padding="max_length"' in utils_text

    print({
        "synthtexteval_revision": SYNTHTEXTEVAL_REVISION,
        "encode_plus_replacements": encode_plus_count,
        "legacy_padding_replacements": legacy_padding_count,
        "source": str(SYNTHTEXTEVAL_DIR),
    })

    # Install repository requirements first, then pin Transformers last so an unpinned
    # transformers requirement cannot leave a v5 installation active.
    run_checked([sys.executable, "-m", "pip", "uninstall", "-y", "SynthTextEval"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "-r", SYNTHTEXTEVAL_DIR / "requirements.txt"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "pyLDAvis", "accelerate>=0.26.0"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", SYNTHTEXTEVAL_DIR])
    run_checked([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

    INSTALL_SENTINEL.write_text(SYNTHTEXTEVAL_REVISION + "\n", encoding="utf-8")
    SYNTH_INSTALL_RESTART_REQUIRED = True
    print("\nSynthTextEval installation/patch completed.")
    print("Restart the runtime ONCE now, then rerun the notebook from the top.")
else:
    SYNTH_INSTALL_RESTART_REQUIRED = False
    print("Pinned SynthTextEval installation already prepared in this runtime; reinstall skipped.")


In [ ]:
# Run after restarting the kernel/runtime. Verify that Python is using the patched clone.
from pathlib import Path
import inspect
import synthtexteval
import transformers

from synthtexteval.eval.descriptive.descriptor import TextDescriptor
from synthtexteval.eval.descriptive.arguments import TextDescriptorArgs
from synthtexteval.eval.downstream.classify import utils as classifier_utils
from synthtexteval.eval.text_quality import metrics as text_quality_metrics

loaded_utils_path = Path(classifier_utils.__file__).resolve()
expected_source_root = Path("/content/synthtexteval-src").resolve()

print("SynthTextEval:", synthtexteval.__file__)
print("Transformers:", transformers.__version__)
print("Classifier utils:", loaded_utils_path)

assert expected_source_root in loaded_utils_path.parents, (
    f"Wrong SynthTextEval copy loaded: {loaded_utils_path}. Expected the editable clone under {expected_source_root}."
)
assert transformers.__version__.startswith("4."), "This notebook expects Transformers 4.x."

tokenize_source = inspect.getsource(classifier_utils.tokenize_data)
assert "encode_plus" not in tokenize_source, "Old encode_plus call is still active."
assert "pad_to_max_length" not in tokenize_source, "Old pad_to_max_length argument is still active."
assert "padding" in tokenize_source and "max_length" in tokenize_source

# The public README uses QualEval, but tolerate an older class name if a repository
# revision exposes it instead. Cell 17 uses the same resolution logic.
quality_class = getattr(text_quality_metrics, "QualEval", None)
if quality_class is None:
    quality_class = getattr(text_quality_metrics, "TextQualityEval", None)
assert quality_class is not None, (
    "No supported text-quality evaluator class (QualEval/TextQualityEval) was found in "
    f"{text_quality_metrics.__file__}."
)
print("Text-quality evaluator:", quality_class.__name__)
print("SynthTextEval installation/import checks passed.")


In [ ]:
import hashlib
import json
import pandas as pd


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


text_records = pd.read_csv(TEXT_RECORDS_CSV)
required_columns = {
    SOURCE_COLUMN, GROUP_COLUMN, COHORT_COLUMN, TEXT_COLUMN, "pair_id", "round_id",
    "source_child_id", "td_child_id", "sli_child_id",
}
missing_columns = required_columns.difference(text_records.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"
assert not text_records[TEXT_COLUMN].isna().any()
assert (text_records[TEXT_COLUMN].astype(str).str.strip() != "").all()

# Preserve the raw export and make the analytic exclusion reproducible. This excludes only
# the known LLM self-commentary, never human ENNI text.
meta_response_mask = pd.Series(False, index=text_records.index)
for marker in META_RESPONSE_MARKERS:
    meta_response_mask |= text_records[TEXT_COLUMN].str.contains(marker, case=False, regex=False, na=False)
meta_response_mask &= text_records[SOURCE_COLUMN].eq("synthetic")
excluded_records = text_records.loc[meta_response_mask].copy()
analysis_records = text_records.loc[~meta_response_mask].copy()
excluded_records.to_csv(RESULTS_DIR / "excluded_meta_generations.csv", index=False)

summary = {
    "raw_records": int(len(text_records)),
    "analytic_records": int(len(analysis_records)),
    "excluded_meta_generations": int(len(excluded_records)),
    "exclusion_markers": list(META_RESPONSE_MARKERS),
    "text_records_sha256": sha256_file(TEXT_RECORDS_CSV),
    "gillam_external_text_sha256": sha256_file(GILLAM_EXTERNAL_TEXT_CSV),
}
(RESULTS_DIR / "input_qc_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(summary)

show_columns = [
    col for col in ["source_csv", "source_row_number", "pair_id", "story_id", "stage"]
    if col in excluded_records.columns
]
if show_columns:
    display(excluded_records[show_columns])


In [ ]:
# Balanced, independently sampled corpus inputs for descriptive and text-quality modules.
# Row i in the synthetic table is NOT assumed to be semantically paired with row i in the real table.
manifest_columns = [
    col for col in [
        TEXT_COLUMN, COHORT_COLUMN, GROUP_COLUMN, "source_child_id", "pair_id", "round_id",
        "source_csv", "source_row_number", "story_id", "stage",
    ]
    if col in analysis_records.columns
]

synthetic_parts, reference_parts = [], []
for cohort_year in COMPARISON_COHORTS:
    for group in COMPARISON_GROUPS:
        synthetic_group = analysis_records.loc[
            (analysis_records[SOURCE_COLUMN] == "synthetic")
            & (analysis_records[COHORT_COLUMN] == cohort_year)
            & (analysis_records[GROUP_COLUMN] == group),
            manifest_columns,
        ]
        reference_group = analysis_records.loc[
            (analysis_records[SOURCE_COLUMN] == "real")
            & (analysis_records[COHORT_COLUMN] == cohort_year)
            & (analysis_records[GROUP_COLUMN] == group),
            manifest_columns,
        ]
        n = min(len(synthetic_group), len(reference_group))
        if MAX_SAMPLES_PER_GROUP is not None:
            n = min(n, MAX_SAMPLES_PER_GROUP)
        assert n > 0, f"No comparable {cohort_year}-year-old {group} text records found."
        seed = SAMPLE_SEED + 100 * cohort_year + (0 if group == "TD" else 1)
        synthetic_parts.append(synthetic_group.sample(n=n, random_state=seed).reset_index(drop=True))
        reference_parts.append(reference_group.sample(n=n, random_state=seed).reset_index(drop=True))

synthetic_texts = pd.concat(synthetic_parts, ignore_index=True)
reference_texts = pd.concat(reference_parts, ignore_index=True)
comparison_df = pd.DataFrame({
    "source": synthetic_texts[TEXT_COLUMN].astype(str).tolist(),
    "reference": reference_texts[TEXT_COLUMN].astype(str).tolist(),
})

# Save exact sampled inputs, not only group counts, so the distributional analyses are reproducible.
synthetic_texts.to_csv(RESULTS_DIR / "balanced_synthetic_sample.csv", index=False)
reference_texts.to_csv(RESULTS_DIR / "balanced_real_sample.csv", index=False)
comparison_df.to_csv(RESULTS_DIR / "balanced_text_quality_input.csv", index=False)

comparison_manifest = pd.concat(
    [
        synthetic_texts.drop(columns=[TEXT_COLUMN], errors="ignore").add_prefix("synthetic_"),
        reference_texts.drop(columns=[TEXT_COLUMN], errors="ignore").add_prefix("reference_"),
    ],
    axis=1,
)
comparison_manifest.to_csv(RESULTS_DIR / "balanced_corpus_sample_manifest.csv", index=False)
print({"synthetic_texts": len(synthetic_texts), "reference_texts": len(reference_texts)})
display(
    synthetic_texts.groupby([COHORT_COLUMN, GROUP_COLUMN]).size().rename("synthetic_n").reset_index()
    .merge(
        reference_texts.groupby([COHORT_COLUMN, GROUP_COLUMN]).size().rename("real_n").reset_index(),
        on=[COHORT_COLUMN, GROUP_COLUMN],
        how="outer",
    )
)


## Descriptive analysis

These are generic corpus-level comparisons. The matching is balanced by cohort and literal corpus group, but row order does not define a child-, prompt-, or story-level pair.

In [ ]:
import os
import importlib.util
import synthtexteval

print("cwd:", os.getcwd())
print("module:", synthtexteval)
print("file:", getattr(synthtexteval, "__file__", None))
print("path:", list(getattr(synthtexteval, "__path__", [])))
print("eval spec:", importlib.util.find_spec("synthtexteval.eval"))

assert importlib.util.find_spec("synthtexteval.eval") is not None, (
    "synthtexteval.eval is still unavailable. Restart the runtime and rerun "
    "the installation cells from the top."
)


In [ ]:
import io
from contextlib import redirect_stdout
import nltk
import pandas as pd
from synthtexteval.eval.descriptive.descriptor import TextDescriptor
from synthtexteval.eval.descriptive.arguments import TextDescriptorArgs

nltk.download("punkt_tab")

synth_df = comparison_df.rename(columns={"source": "output_text"})[["output_text"]].copy()
real_texts = comparison_df["reference"].tolist()
descriptor_args = TextDescriptorArgs(
    pkl_file_path=str(RESULTS_DIR / "entity-output.pkl"),
    plt_file_path=str(RESULTS_DIR / "entity-analysis.png"),
    produce_plot=True,
)
desc_analyze = TextDescriptor(
    texts=synth_df["output_text"].tolist(),
    args=descriptor_args,
    reference_texts=real_texts,
)

# The package prints these metrics rather than consistently returning a structured object.
# Capture the exact printed output while still showing it in the notebook and save it.
descriptive_stdout = io.StringIO()
with redirect_stdout(descriptive_stdout):
    descriptive_result = desc_analyze._compare_to_reference_distribution(
        metrics=["text-length", "jaccard", "cosine"]
    )
descriptive_text = descriptive_stdout.getvalue()
print(descriptive_text, end="")
(RESULTS_DIR / "descriptive_distribution_comparison.txt").write_text(
    descriptive_text, encoding="utf-8"
)
if descriptive_result is not None:
    (RESULTS_DIR / "descriptive_distribution_return_value.txt").write_text(
        repr(descriptive_result) + "\n", encoding="utf-8"
    )

# Optional interactive topic display. This describes the synthetic corpus; it should not
# be interpreted as a real-vs-synthetic topic-distribution comparison by itself.
import pyLDAvis
pyLDAvis.enable_notebook()
topic_model = desc_analyze._topic_modeling_display(num_topics=3)
if topic_model is not None:
    try:
        pyLDAvis.save_html(topic_model, str(RESULTS_DIR / "synthetic_lda_topics.html"))
    except Exception as exc:
        print("Could not save LDA HTML:", exc)
topic_model


## Downstream utility: rotating synthetic-to-real TD/SLI classification

For each usable generation round, the notebook derives pair membership from the IDs embedded in the generated artifacts. Synthetic descendants of prompt pairs and real prompt-pair narratives form the training material; one whole prompt pair per cohort is reserved for validation; real narratives from all other paired children form the test set. Split membership uses `(cohort_year, pair_id)` rather than bare `pair_id`, and train/validation/test child sets are asserted to be disjoint within each round.

The stock SynthTextEval random split helper is deliberately not used because it could split related prompt-pair descendants across partitions. The notebook still uses SynthTextEval's classifier and fairness utilities.

The four trained round-models are treated as repeated rotating-split evaluations. A child may legitimately be tested by more than one round-model, so **raw predictions from different rounds are never pooled to create an inflated sample size**. Accuracy/F1 and cohort fairness/performance are computed within each training round and then summarized across rounds. The reported SD therefore describes variation across the rotating trained models; it is not an independence-based standard error or confidence interval.

The same trained models can additionally be tested on Gillam as a strictly external corpus. Gillam contributes no training or validation rows, and an ID-disjointness assertion is enforced. Each Gillam child is evaluated once per trained round-model; external metrics are therefore computed per model and summarized across models rather than treating the four predictions per child as four independent children. Because each Gillam row is a complete child transcript whereas the internal ENNI analysis is story-level, this is an external-transfer stress test rather than a like-for-like internal classification estimate.


In [ ]:
# Build round-specific datasets from generated-artifact IDs using the composite
# (cohort_year, pair_id) key throughout.
CLASSIFICATION_DIR = RESULTS_DIR / "classification"
CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)

synthetic_source = analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("synthetic")].copy()
real_source = analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("real")].copy()
for column in ("source_child_id", "td_child_id", "sli_child_id"):
    synthetic_source[column] = pd.to_numeric(synthetic_source[column], errors="raise").astype("int64")
real_source["source_child_id"] = pd.to_numeric(real_source["source_child_id"], errors="raise").astype("int64")


def add_pair_key(frame):
    frame = frame.copy()
    frame["_pair_key"] = list(zip(
        frame[COHORT_COLUMN].astype(int),
        frame["pair_id"].astype(str),
    ))
    return frame


synthetic_source = add_pair_key(synthetic_source)

# Gillam is a separate external test source. Each row is one complete child transcript;
# it is never concatenated with analysis_records or any training frame.
gillam_records = pd.read_csv(GILLAM_EXTERNAL_TEXT_CSV)
gillam_required = {TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "source", "sex"}
gillam_missing = gillam_required.difference(gillam_records.columns)
assert not gillam_missing, f"Missing Gillam columns: {sorted(gillam_missing)}"
assert gillam_records["source"].eq("real_external").all()
assert set(gillam_records[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
assert gillam_records["source_child_id"].nunique() == len(gillam_records)
assert gillam_records[TEXT_COLUMN].notna().all()
gillam_ids = set(gillam_records["source_child_id"].astype(str))
enni_ids = set(real_source["source_child_id"].astype(str))
external_id_overlap = sorted(gillam_ids.intersection(enni_ids))
assert not external_id_overlap, f"Gillam external IDs occur in ENNI data: {external_id_overlap}"

external_columns = [TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "source_dataset", "sex"]
external_gillam_test = gillam_records[external_columns].copy()
external_gillam_test = external_gillam_test.rename(columns={TEXT_COLUMN: "output_text"})
external_gillam_test["Label"] = external_gillam_test[GROUP_COLUMN].map(LABEL_MAPPING)
external_gillam_test["classification_round"] = "external_gillam"  # dataset identifier, not model round
external_gillam_test["pair_id"] = pd.NA
assert external_gillam_test["Label"].notna().all()

EXTERNAL_GILLAM_DIR = CLASSIFICATION_DIR / "external_gillam"
EXTERNAL_GILLAM_DIR.mkdir(parents=True, exist_ok=True)
external_gillam_test.to_csv(EXTERNAL_GILLAM_DIR / "test.csv", index=False)
external_gillam_audit = {
    "status": "prepared_test_only",
    "rows": int(len(external_gillam_test)),
    "unique_children": int(external_gillam_test["source_child_id"].nunique()),
    "group_counts": external_gillam_test[GROUP_COLUMN].value_counts().to_dict(),
    "enni_id_overlap": len(external_id_overlap),
    "document_unit": "one complete Gillam child transcript",
    "training_use": False,
}
(EXTERNAL_GILLAM_DIR / "input_audit.json").write_text(json.dumps(external_gillam_audit, indent=2) + "\n")
print({"external_gillam": external_gillam_audit})

pair_metadata = synthetic_source[[COHORT_COLUMN, "pair_id", "td_child_id", "sli_child_id", "_pair_key"]].drop_duplicates()
assert pair_metadata.groupby([COHORT_COLUMN, "pair_id"]).size().eq(1).all(), "Ambiguous generated pair metadata."
pair_rounds = synthetic_source[[COHORT_COLUMN, "pair_id", "round_id", "_pair_key"]].drop_duplicates()
assert pair_rounds.groupby([COHORT_COLUMN, "pair_id"])["round_id"].nunique().eq(1).all(), "A generated pair occurs in multiple rounds."

# Audit whether raw pair_id values are reused across cohorts. Reuse is safe because all
# split operations below use _pair_key=(cohort_year, pair_id).
pair_id_cohort_counts = pair_metadata.groupby("pair_id")[COHORT_COLUMN].nunique().rename("n_cohorts").reset_index()
pair_id_cohort_counts.to_csv(CLASSIFICATION_DIR / "pair_id_cross_cohort_audit.csv", index=False)
print("pair_id values reused across cohorts:", int((pair_id_cohort_counts["n_cohorts"] > 1).sum()))

sli_pair_children = pair_metadata[[COHORT_COLUMN, "pair_id", "_pair_key", "sli_child_id"]].rename(columns={"sli_child_id": "source_child_id"})
sli_pair_children[GROUP_COLUMN] = "SLI"
td_pair_children = pair_metadata[[COHORT_COLUMN, "pair_id", "_pair_key", "td_child_id"]].rename(columns={"td_child_id": "source_child_id"})
td_pair_children[GROUP_COLUMN] = "TD"
pair_children = pd.concat([sli_pair_children, td_pair_children], ignore_index=True)
assert not pair_children.duplicated([COHORT_COLUMN, "pair_id", GROUP_COLUMN]).any()

real_columns = [TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id"]
real_source = real_source[real_columns].copy()


def real_rows_for_pair_keys(pair_keys):
    pair_lookup = pair_children.loc[
        pair_children["_pair_key"].isin(pair_keys),
        [COHORT_COLUMN, GROUP_COLUMN, "source_child_id", "pair_id", "_pair_key"],
    ].copy()
    rows = real_source.merge(
        pair_lookup,
        on=[COHORT_COLUMN, GROUP_COLUMN, "source_child_id"],
        how="inner",
        validate="many_to_one",
    )
    return rows


def child_keys(frame):
    return set(zip(
        frame[COHORT_COLUMN].astype(int),
        frame[GROUP_COLUMN].astype(str),
        frame["source_child_id"].astype(int),
    ))


def classifier_frame(frame, classification_round):
    output = frame[[TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "pair_id"]].copy()
    output = output.rename(columns={TEXT_COLUMN: "output_text"})
    output["Label"] = output[GROUP_COLUMN].map(LABEL_MAPPING)
    output["classification_round"] = classification_round
    assert output["Label"].notna().all()
    return output


prepared_rounds, split_audit = {}, []
for round_id in CLASSIFICATION_ROUNDS:
    round_synthetic = synthetic_source.loc[synthetic_source["round_id"].eq(round_id)].copy()
    if round_synthetic.empty:
        split_audit.append({"round_id": round_id, "status": "skipped_no_generated_rows"})
        continue

    train_pair_keys, validation_pair_keys, test_pair_keys = set(), set(), set()
    usable_cohorts = []

    for cohort_year in sorted(round_synthetic[COHORT_COLUMN].unique()):
        prompt_pair_keys = sorted(set(
            round_synthetic.loc[round_synthetic[COHORT_COLUMN].eq(cohort_year), "_pair_key"]
        ))
        all_pair_keys = set(pair_metadata.loc[pair_metadata[COHORT_COLUMN].eq(cohort_year), "_pair_key"])
        if len(prompt_pair_keys) < 2:
            split_audit.append({
                "round_id": round_id,
                "cohort_year": int(cohort_year),
                "status": "skipped_one_prompt_pair",
            })
            continue

        validation_pair_key = prompt_pair_keys[-1]  # deterministic whole-pair validation split
        train_pair_keys.update(set(prompt_pair_keys) - {validation_pair_key})
        validation_pair_keys.add(validation_pair_key)
        test_pair_keys.update(all_pair_keys - set(prompt_pair_keys))
        usable_cohorts.append(int(cohort_year))

    if not usable_cohorts:
        continue

    assert train_pair_keys.isdisjoint(validation_pair_keys)
    assert train_pair_keys.isdisjoint(test_pair_keys)
    assert validation_pair_keys.isdisjoint(test_pair_keys)

    synthetic_train = round_synthetic.loc[round_synthetic["_pair_key"].isin(train_pair_keys)].copy()
    real_train = real_rows_for_pair_keys(train_pair_keys)
    real_validation = real_rows_for_pair_keys(validation_pair_keys)
    real_test = real_rows_for_pair_keys(test_pair_keys)

    real_train = real_train.loc[real_train[COHORT_COLUMN].isin(usable_cohorts)]
    real_validation = real_validation.loc[real_validation[COHORT_COLUMN].isin(usable_cohorts)]
    real_test = real_test.loc[real_test[COHORT_COLUMN].isin(usable_cohorts)]

    # Child-level separation within a training round.
    train_children = child_keys(real_train)
    validation_children = child_keys(real_validation)
    test_children = child_keys(real_test)
    assert train_children.isdisjoint(validation_children), f"Train/validation child overlap in {round_id}."
    assert train_children.isdisjoint(test_children), f"Train/test child overlap in {round_id}."
    assert validation_children.isdisjoint(test_children), f"Validation/test child overlap in {round_id}."

    usable_prompt_keys = train_pair_keys | validation_pair_keys
    prompt_children = child_keys(pair_children.loc[pair_children["_pair_key"].isin(usable_prompt_keys)])
    assert not test_children.intersection(prompt_children), f"Prompt/test child overlap in {round_id}."

    assert set(synthetic_train[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_train[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_validation[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_test[GROUP_COLUMN]) == set(COMPARISON_GROUPS)

    round_dir = CLASSIFICATION_DIR / round_id
    round_dir.mkdir(parents=True, exist_ok=True)
    frames = {
        "train": classifier_frame(real_train, round_id),
        "validation": classifier_frame(real_validation, round_id),
        "synthetic": classifier_frame(synthetic_train, round_id),
        "test": classifier_frame(real_test, round_id),
    }
    for split_name, frame in frames.items():
        frame.to_csv(round_dir / f"{split_name}.csv", index=False)
    (round_dir / "label_mapping.json").write_text(json.dumps(LABEL_MAPPING, indent=2) + "\n")

    prepared_rounds[round_id] = {"path": round_dir, **frames}
    split_audit.append({
        "round_id": round_id,
        "status": "prepared",
        "cohorts": usable_cohorts,
        "train_pair_keys": len(train_pair_keys),
        "validation_pair_keys": len(validation_pair_keys),
        "test_pair_keys": len(test_pair_keys),
        "real_train_rows": len(frames["train"]),
        "real_validation_rows": len(frames["validation"]),
        "synthetic_train_rows": len(frames["synthetic"]),
        "real_test_rows": len(frames["test"]),
        "train_validation_child_overlap": 0,
        "train_test_child_overlap": 0,
        "validation_test_child_overlap": 0,
        "test_prompt_child_overlap": 0,
        "pair_mapping_source": "generated_artifact_metadata",
        "split_key": "(cohort_year, pair_id)",
    })

split_audit_df = pd.DataFrame(split_audit)
split_audit_df.to_csv(CLASSIFICATION_DIR / "split_audit.csv", index=False)
display(split_audit_df)
assert prepared_rounds, "No usable round-specific classification datasets were prepared."


In [ ]:
# Fine-tune and test SynthTextEval classifiers. Each prediction row records BOTH the
# evaluation dataset identifier and the training round/model that produced it.
if RUN_CLASSIFICATION:
    import torch
    from synthtexteval.eval.downstream.classify.train_classifier import (
        Arguments, Classifier, ModelArguments, TrainingArguments,
    )

    assert torch.cuda.is_available(), "Enable a Colab GPU before running classifier training."
    prediction_frames = []
    training_settings = ("real-only", "synthetic-train-only", "synthetic-train-augment")

    for round_id, prepared in prepared_rounds.items():
        round_dir = prepared["path"]
        for training_setting in training_settings:
            setting_dir = round_dir / training_setting
            model_dir = setting_dir / "model"
            setting_dir.mkdir(parents=True, exist_ok=True)

            train_args = TrainingArguments(
                output_dir=str(setting_dir / "trainer"),
                num_train_epochs=3,
                per_device_train_batch_size=8,
                per_device_eval_batch_size=8,
                save_strategy="no",
                report_to=[],
            )
            model_args = ModelArguments(
                model_name=CLASSIFIER_MODEL,
                path_to_dataset=str(round_dir),
                path_to_model=str(model_dir),
                n_labels=len(LABEL_MAPPING),
                problem_type="single_label_classification",
                text_field="output_text",
                label_field="Label",
                synthetic_usage=training_setting,
                is_train=True,
            )
            Classifier(Arguments(train=train_args, model=model_args)).finetune_model()

            prediction_path = setting_dir / "test_predictions.csv"
            test_args = TrainingArguments(
                output_dir=str(setting_dir / "tester"),
                per_device_eval_batch_size=8,
                save_strategy="no",
                report_to=[],
            )
            test_model_args = ModelArguments(
                model_name=CLASSIFIER_MODEL,
                path_to_dataset=str(round_dir / "test.csv"),
                path_to_model=str(model_dir),
                path_to_output_csv=str(prediction_path),
                path_to_aggregated_results=str(setting_dir / "test_metrics.csv"),
                n_labels=len(LABEL_MAPPING),
                problem_type="single_label_classification",
                text_field="output_text",
                label_field="Label",
                retain_columns=["cohort_year", "classification_round", "group", "source_child_id", "pair_id"],
                is_test=True,
            )
            Classifier(Arguments(train=test_args, model=test_model_args)).test_model()
            prediction_frame = pd.read_csv(prediction_path)
            prediction_frame["training_setting"] = training_setting
            prediction_frame["training_round"] = round_id
            prediction_frame["evaluation_set"] = "internal_enni"
            prediction_frames.append(prediction_frame)

            if RUN_GILLAM_EXTERNAL_TEST:
                external_prediction_path = setting_dir / "external_gillam_predictions.csv"
                external_test_args = TrainingArguments(
                    output_dir=str(setting_dir / "external_gillam_tester"),
                    per_device_eval_batch_size=8,
                    save_strategy="no",
                    report_to=[],
                )
                external_model_args = ModelArguments(
                    model_name=CLASSIFIER_MODEL,
                    path_to_dataset=str(EXTERNAL_GILLAM_DIR / "test.csv"),
                    path_to_model=str(model_dir),
                    path_to_output_csv=str(external_prediction_path),
                    path_to_aggregated_results=str(setting_dir / "external_gillam_metrics.csv"),
                    n_labels=len(LABEL_MAPPING),
                    problem_type="single_label_classification",
                    text_field="output_text",
                    label_field="Label",
                    retain_columns=["cohort_year", "classification_round", "group", "source_child_id", "source_dataset", "sex"],
                    is_test=True,
                )
                Classifier(Arguments(train=external_test_args, model=external_model_args)).test_model()
                external_prediction_frame = pd.read_csv(external_prediction_path)
                external_prediction_frame["training_setting"] = training_setting
                external_prediction_frame["training_round"] = round_id
                external_prediction_frame["evaluation_set"] = "external_gillam"
                prediction_frames.append(external_prediction_frame)

    classification_predictions = pd.concat(prediction_frames, ignore_index=True)
    classification_predictions.to_csv(CLASSIFICATION_DIR / "all_round_predictions.csv", index=False)
    display(
        classification_predictions
        .groupby(["evaluation_set", "training_setting", "training_round", "Ground"])
        .size().rename("n_predictions").reset_index()
    )
else:
    print("Datasets prepared. Set RUN_CLASSIFICATION = True in a GPU runtime to train and test the models.")


In [ ]:
# Round-wise downstream utility and fairness audit.
# Do NOT pool repeated predictions from multiple trained round-models as if they were
# independent observations. Compute metrics within each round, then summarize across rounds.
if "classification_predictions" in globals():
    from sklearn.metrics import accuracy_score, f1_score
    from synthtexteval.eval.fairness.metrics import analyze_group_fairness_performance

    internal_predictions = classification_predictions.loc[
        classification_predictions["evaluation_set"].eq("internal_enni")
    ].copy()

    internal_overall_rows = []
    internal_subgroup_performance_parts = []
    internal_fairness_parts = []

    for (training_setting, training_round), prediction_frame in internal_predictions.groupby(
        ["training_setting", "training_round"], sort=True
    ):
        internal_overall_rows.append({
            "training_setting": training_setting,
            "training_round": training_round,
            "n_predictions": int(len(prediction_frame)),
            "n_unique_children": int(prediction_frame["source_child_id"].nunique()),
            "accuracy": float(accuracy_score(prediction_frame["Ground"], prediction_frame["Predicted"])),
            "f1_macro": float(f1_score(
                prediction_frame["Ground"], prediction_frame["Predicted"],
                average="macro", zero_division=0,
            )),
            "f1_micro": float(f1_score(
                prediction_frame["Ground"], prediction_frame["Predicted"],
                average="micro", zero_division=0,
            )),
        })

        performance, fairness = analyze_group_fairness_performance(
            prediction_frame,
            problem_type="multiclass",
            num_classes=len(LABEL_MAPPING),
            subgroup_type=COHORT_COLUMN,
        )
        performance = performance.copy()
        fairness = fairness.copy()
        performance["training_setting"] = training_setting
        performance["training_round"] = training_round
        fairness["training_setting"] = training_setting
        fairness["training_round"] = training_round
        internal_subgroup_performance_parts.append(performance)
        internal_fairness_parts.append(fairness)

    internal_overall_round_metrics = pd.DataFrame(internal_overall_rows)
    internal_overall_round_metrics.to_csv(
        CLASSIFICATION_DIR / "internal_overall_round_metrics.csv", index=False
    )
    internal_overall_summary = (
        internal_overall_round_metrics
        .groupby("training_setting", as_index=False)
        .agg(
            n_rounds=("training_round", "nunique"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_sd=("accuracy", "std"),
            f1_macro_mean=("f1_macro", "mean"),
            f1_macro_sd=("f1_macro", "std"),
            f1_micro_mean=("f1_micro", "mean"),
            f1_micro_sd=("f1_micro", "std"),
            test_predictions_per_round_mean=("n_predictions", "mean"),
            unique_children_per_round_mean=("n_unique_children", "mean"),
        )
    )
    internal_overall_summary.to_csv(
        CLASSIFICATION_DIR / "internal_overall_summary_across_rounds.csv", index=False
    )

    internal_subgroup_performance = pd.concat(
        internal_subgroup_performance_parts, ignore_index=True
    )
    internal_subgroup_performance.to_csv(
        CLASSIFICATION_DIR / "internal_cohort_performance_by_round.csv", index=False
    )

    perf_required = {"Accuracy", "Group Type", "Num of Samples", "f1_macro", "f1_micro"}
    if perf_required.issubset(internal_subgroup_performance.columns):
        internal_cohort_summary = (
            internal_subgroup_performance
            .groupby(["training_setting", "Group Type"], as_index=False)
            .agg(
                n_rounds=("training_round", "nunique"),
                accuracy_mean=("Accuracy", "mean"),
                accuracy_sd=("Accuracy", "std"),
                f1_macro_mean=("f1_macro", "mean"),
                f1_macro_sd=("f1_macro", "std"),
                f1_micro_mean=("f1_micro", "mean"),
                f1_micro_sd=("f1_micro", "std"),
                samples_per_round_mean=("Num of Samples", "mean"),
                samples_per_round_min=("Num of Samples", "min"),
                samples_per_round_max=("Num of Samples", "max"),
            )
        )
        internal_cohort_summary.to_csv(
            CLASSIFICATION_DIR / "internal_cohort_summary_across_rounds.csv", index=False
        )
    else:
        internal_cohort_summary = pd.DataFrame()

    internal_round_fairness = pd.concat(internal_fairness_parts, ignore_index=True)
    internal_round_fairness.to_csv(
        CLASSIFICATION_DIR / "internal_cohort_fairness_by_round.csv", index=False
    )

    # Only summarize columns whose names clearly identify them as fairness measures;
    # leave possible numeric group/class identifiers untouched.
    fairness_tokens = ("fair", "equal", "odds", "difference", "gap", "ratio", "score")
    fairness_metric_columns = [
        col for col in internal_round_fairness.select_dtypes(include="number").columns
        if any(token in col.lower() for token in fairness_tokens)
    ]
    if fairness_metric_columns:
        fairness_summary = (
            internal_round_fairness
            .groupby("training_setting")[fairness_metric_columns]
            .agg(["mean", "std"])
        )
        fairness_summary.columns = [f"{metric}_{stat}" for metric, stat in fairness_summary.columns]
        fairness_summary = fairness_summary.reset_index()
        fairness_summary.to_csv(
            CLASSIFICATION_DIR / "internal_fairness_summary_across_rounds.csv", index=False
        )
    else:
        fairness_summary = pd.DataFrame()

    print("Internal overall metrics by round")
    display(internal_overall_round_metrics)
    print("Internal overall mean ± SD across trained rounds")
    display(internal_overall_summary)
    print("Cohort performance by round")
    display(internal_subgroup_performance)
    if not internal_cohort_summary.empty:
        print("Cohort performance mean ± SD across trained rounds")
        display(internal_cohort_summary)
    print("Fairness output by round (not pooled)")
    display(internal_round_fairness)
    if not fairness_summary.empty:
        print("Fairness metric mean ± SD across trained rounds")
        display(fairness_summary)

    if RUN_GILLAM_EXTERNAL_TEST:
        external_predictions = classification_predictions.loc[
            classification_predictions["evaluation_set"].eq("external_gillam")
        ].copy()
        external_predictions.to_csv(
            CLASSIFICATION_DIR / "all_external_gillam_predictions.csv", index=False
        )

        external_round_rows = []
        for (training_setting, training_round), frame in external_predictions.groupby(
            ["training_setting", "training_round"], sort=True
        ):
            external_round_rows.append({
                "training_setting": training_setting,
                "training_round": training_round,
                "n_predictions": int(len(frame)),
                "n_unique_children": int(frame["source_child_id"].nunique()),
                "accuracy": float(accuracy_score(frame["Ground"], frame["Predicted"])),
                "f1_macro": float(f1_score(
                    frame["Ground"], frame["Predicted"], average="macro", zero_division=0
                )),
                "f1_micro": float(f1_score(
                    frame["Ground"], frame["Predicted"], average="micro", zero_division=0
                )),
            })

        external_round_metrics = pd.DataFrame(external_round_rows)
        external_round_metrics.to_csv(
            CLASSIFICATION_DIR / "external_gillam_round_metrics.csv", index=False
        )
        external_summary = (
            external_round_metrics
            .groupby("training_setting", as_index=False)
            .agg(
                n_rounds=("training_round", "nunique"),
                accuracy_mean=("accuracy", "mean"),
                accuracy_sd=("accuracy", "std"),
                f1_macro_mean=("f1_macro", "mean"),
                f1_macro_sd=("f1_macro", "std"),
                f1_micro_mean=("f1_micro", "mean"),
                f1_micro_sd=("f1_micro", "std"),
                unique_children=("n_unique_children", "max"),
            )
        )
        external_summary.to_csv(
            CLASSIFICATION_DIR / "external_gillam_summary_across_rounds.csv", index=False
        )

        print("Gillam external metrics by trained round-model")
        display(external_round_metrics)
        print("Gillam external mean ± SD across trained round-models")
        display(external_summary)
else:
    print("Run the classifier cell first; no prediction file has been created.")


## Privacy and memorization

This section restores SynthTextEval's package-native `entity_leakage` and `search_and_compute_EPO` analyses. The package expects a user-supplied list of private entities, but this project has no such list because the stories are de-identified. We therefore extract candidate named entities from the balanced real corpus and report their occurrence and surrounding-span overlap in the synthetic corpus. These are reference-overlap diagnostics—not verified private-entity leakage rates. Shared ENNI character names, places, and NER errors can produce matches.

Canary attacks are not applicable retrospectively: they require known secrets to have been deliberately inserted during generator training. No canaries were planted in the model used to produce these narratives.

In [ ]:
if RUN_PRIVACY_ANALYSIS:
    from collections import Counter, defaultdict
    import spacy
    from synthtexteval.eval.privacy.metrics import entity_leakage, search_and_compute_EPO

    PRIVACY_DIR = RESULTS_DIR / "privacy"
    PRIVACY_DIR.mkdir(parents=True, exist_ok=True)
    # Privacy/reference-overlap analysis should use the full available corpora, not the
    # balanced descriptive subsample. If the generator was trained/prompted from a narrower
    # known source corpus, replace real_privacy_texts with that exact source set.
    synthetic_privacy_texts = (
        analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("synthetic"), TEXT_COLUMN]
        .dropna().astype(str).tolist()
    )
    real_privacy_texts = (
        analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("real"), TEXT_COLUMN]
        .dropna().astype(str).tolist()
    )

    # The toolkit requires private entities supplied by the user. No protected-entity list
    # exists here, so use NER candidates from real ENNI and retain their provenance.
    nlp = spacy.load("en_core_web_sm")
    entity_document_frequency = Counter()
    entity_labels = defaultdict(Counter)
    entity_surface = {}
    for document in nlp.pipe(real_privacy_texts, batch_size=64):
        seen_in_document = set()
        for entity in document.ents:
            surface = " ".join(entity.text.split()).strip()
            if entity.label_ not in PRIVACY_ENTITY_LABELS or len(surface) < 2 or surface.isdigit():
                continue
            key = surface.casefold()
            entity_surface.setdefault(key, surface)
            entity_labels[key][entity.label_] += 1
            if key not in seen_in_document:
                entity_document_frequency[key] += 1
                seen_in_document.add(key)

    entity_candidates = [entity_surface[key] for key in sorted(entity_surface)]
    candidate_rows = []
    for key in sorted(entity_surface):
        candidate_rows.append({
            "entity": entity_surface[key],
            "ner_labels": "|".join(sorted(entity_labels[key])),
            "real_document_frequency": int(entity_document_frequency[key]),
            "verified_private_entity": False,
        })
    pd.DataFrame(candidate_rows, columns=[
        "entity", "ner_labels", "real_document_frequency", "verified_private_entity"
    ]).to_csv(PRIVACY_DIR / "reference_named_entity_candidates.csv", index=False)

    if entity_candidates:
        average_entity_matches, entity_details = entity_leakage(
            paragraphs=synthetic_privacy_texts,
            entities=entity_candidates,
            entity_leakage_result_path=str(PRIVACY_DIR / "entity_leakage.pkl"),
        )
        leakage_rows = []
        leaked_entity_union = set()
        for row_index, text in enumerate(synthetic_privacy_texts):
            entity_flags, match_count = entity_details[text]
            matched_entities = sorted(entity for entity, found in entity_flags.items() if found)
            leaked_entity_union.update(matched_entities)
            leakage_rows.append({
                "comparison_row": row_index,
                "candidate_entity_match_count": int(match_count),
                "matched_candidate_entities": "|".join(matched_entities),
            })
        pd.DataFrame(leakage_rows).to_csv(
            PRIVACY_DIR / "synthetic_entity_matches.csv", index=False
        )

        synthetic_privacy_frame = pd.DataFrame({"output_text": synthetic_privacy_texts})
        phrase_path = PRIVACY_DIR / "entity_context_spans.csv"
        search_and_compute_EPO(
            synth_file=synthetic_privacy_frame,
            reference_texts=real_privacy_texts,
            synth_phrase_file_path=str(phrase_path),
            entity_patterns=entity_candidates,
            max_window_len=PRIVACY_MAX_CONTEXT_WINDOW,
            text_field="output_text",
        )
        phrase_frame = pd.read_csv(phrase_path)
        if phrase_frame.empty:
            overlap_counts = pd.DataFrame(columns=[
                "Context Length", "candidate_phrase_count",
                "reference_overlap_count", "reference_overlap_rate"
            ])
        else:
            phrase_frame = phrase_frame.drop_duplicates(
                subset=["Entity", "Phrase", "Context Length"]
            ).copy()
            phrase_frame["reference_match"] = phrase_frame["Phrase"].map(
                lambda phrase: any(phrase in reference for reference in real_privacy_texts)
            )
            phrase_frame.to_csv(PRIVACY_DIR / "entity_context_spans_with_matches.csv", index=False)
            overlap_counts = phrase_frame.groupby("Context Length").agg(
                candidate_phrase_count=("Phrase", "size"),
                reference_overlap_count=("reference_match", "sum"),
            ).reset_index()
            overlap_counts["reference_overlap_rate"] = (
                overlap_counts["reference_overlap_count"]
                / overlap_counts["candidate_phrase_count"]
            )
        overlap_counts.to_csv(PRIVACY_DIR / "entity_phrase_overlap_counts.csv", index=False)
    else:
        average_entity_matches = 0.0
        leaked_entity_union = set()
        overlap_counts = pd.DataFrame(columns=[
            "Context Length", "candidate_phrase_count",
            "reference_overlap_count", "reference_overlap_rate"
        ])
        overlap_counts.to_csv(PRIVACY_DIR / "entity_phrase_overlap_counts.csv", index=False)

    privacy_summary = {
        "toolkit_functions": ["entity_leakage", "search_and_compute_EPO"],
        "synthetic_documents": len(synthetic_privacy_texts),
        "real_reference_documents": len(real_privacy_texts),
        "candidate_named_entities": len(entity_candidates),
        "candidate_entities_matched_at_least_once": len(leaked_entity_union),
        "average_candidate_entity_matches_per_synthetic_document": float(average_entity_matches),
        "candidate_entity_source": "spaCy NER over the full available real ENNI reference corpus",
        "verified_private_entity_list_available": False,
        "interpretation": (
            "Reference named-entity and context-span overlap diagnostic only; "
            "not a verified privacy-leakage estimate."
        ),
        "canary_analysis_run": False,
        "canary_reason": "No known canaries were planted during generator training.",
        "epo_case_sensitive_as_implemented_by_package": True,
    }
    (PRIVACY_DIR / "privacy_summary.json").write_text(
        json.dumps(privacy_summary, indent=2) + "\n"
    )
    print(privacy_summary)
    display(overlap_counts)
else:
    print("Privacy analysis skipped because RUN_PRIVACY_ANALYSIS is False.")


## Automated text-quality evaluation

These toolkit outputs compare balanced synthetic and real corpora. They are not clinical validity measures. Perplexity and Fréchet distance can run on CPU; MAUVE should be enabled only in a GPU runtime.

In [ ]:
import dataclasses
import inspect
from synthtexteval.eval.text_quality import metrics as text_quality_metrics
from synthtexteval.eval.text_quality.arguments import (
    Arguments as TextQualityArguments,
    FrechetArgs,
    LMArgs,
    MauveArgs,
)

# Resolve the evaluator class against the actual pinned source loaded by this runtime.
QualityEvalClass = getattr(text_quality_metrics, "QualEval", None)
if QualityEvalClass is None:
    QualityEvalClass = getattr(text_quality_metrics, "TextQualityEval", None)
if QualityEvalClass is None:
    raise ImportError(
        "Neither QualEval nor TextQualityEval exists in "
        f"{text_quality_metrics.__file__}. Inspect that module before continuing."
    )

quality_args = TextQualityArguments(
    frechet=FrechetArgs(source_text_column="source", ref_text_column="reference"),
    mauve=MauveArgs(source_text_column="source", ref_text_column="reference"),
    perplexity=LMArgs(source_text_column="source", ref_text_column="reference"),
)
quality_evaluator = QualityEvalClass(quality_args)

# Save the exact evaluator class, argument values and MAUVE wrapper source so the result can
# be reconstructed even if the upstream package changes later.
try:
    quality_args_serialized = dataclasses.asdict(quality_args)
except TypeError:
    quality_args_serialized = {
        key: vars(value) if hasattr(value, "__dict__") else str(value)
        for key, value in vars(quality_args).items()
    }
quality_method_manifest = {
    "evaluator_class": QualityEvalClass.__name__,
    "metrics_module": text_quality_metrics.__file__,
    "arguments": quality_args_serialized,
}
(RESULTS_DIR / "text_quality_configuration.json").write_text(
    json.dumps(quality_method_manifest, indent=2, default=str) + "\n"
)
try:
    (RESULTS_DIR / "synthtexteval_calculate_mauve_score_source.txt").write_text(
        inspect.getsource(QualityEvalClass.calculate_mauve_score), encoding="utf-8"
    )
except (OSError, TypeError):
    pass

quality_evaluator.calculate_perplexity(comparison_df)
quality_evaluator.calculate_fid_score(comparison_df)
if RUN_MAUVE:
    quality_evaluator.calculate_mauve_score(comparison_df)
quality_results = quality_evaluator.return_results()
quality_evaluator.print_metrics(quality_results)
(RESULTS_DIR / "text_quality_results.json").write_text(
    json.dumps(quality_results, indent=2, default=str) + "\n"
)


### MAUVE stability diagnostics

MAUVE is a distributional estimate and the balanced corpus here is much smaller than the thousands of examples commonly used in MAUVE benchmarking. The notebook therefore reports two supplementary diagnostics using an explicit direct `mauve.compute_mauve` configuration:

1. **Full-corpus seed stability:** every run contains all balanced documents; variation mainly reflects MAUVE's stochastic clustering/seed behavior.
2. **80% subsampling sensitivity:** each run draws a deterministic random subsample without replacement and varies the MAUVE seed; variation reflects both sample composition and MAUVE stochasticity.

These distributions are stability/sensitivity analyses, not formal confidence intervals. Their direct-MAUVE settings are saved separately from the SynthTextEval wrapper configuration so the two are not silently conflated.


In [ ]:
import numpy as np
import mauve


def repeated_mauve(
    real_texts,
    synth_texts,
    n_runs,
    sample_size,
    sampling_seed,
    label,
):
    n = min(len(real_texts), len(synth_texts))
    if sample_size is None:
        sample_size = n
    if not 0 < sample_size <= n:
        raise ValueError(f"sample_size must be between 1 and {n}, got {sample_size}.")
    if n_runs < 1:
        raise ValueError(f"n_runs must be positive, got {n_runs}.")

    rng = np.random.default_rng(sampling_seed)
    rows = []
    for run_index in range(n_runs):
        if sample_size == n:
            real_idx = np.arange(len(real_texts))
            synth_idx = np.arange(len(synth_texts))
        else:
            real_idx = rng.choice(len(real_texts), size=sample_size, replace=False)
            synth_idx = rng.choice(len(synth_texts), size=sample_size, replace=False)

        real_sample = [real_texts[i] for i in real_idx]
        synth_sample = [synth_texts[i] for i in synth_idx]
        mauve_seed = MAUVE_STABILITY_BASE_SEED + run_index

        result = mauve.compute_mauve(
            p_text=real_sample,   # canonical reference/human distribution
            q_text=synth_sample,  # generated/synthetic distribution
            device_id=MAUVE_DEVICE_ID,
            max_text_length=MAUVE_MAX_TEXT_LENGTH,
            seed=mauve_seed,
            verbose=False,
        )
        rows.append({
            "analysis": label,
            "run": run_index,
            "sampling_seed": sampling_seed,
            "mauve_seed": mauve_seed,
            "sample_size_per_distribution": sample_size,
            "mauve": float(result.mauve),
        })

    frame = pd.DataFrame(rows)
    summary = {
        "analysis": label,
        "n_runs": int(len(frame)),
        "sample_size_per_distribution": int(sample_size),
        "full_balanced_corpus_size_per_distribution": int(n),
        "mean": float(frame["mauve"].mean()),
        "sd_sample": float(frame["mauve"].std(ddof=1)) if len(frame) > 1 else 0.0,
        "min": float(frame["mauve"].min()),
        "max": float(frame["mauve"].max()),
        "sampling_seed": int(sampling_seed),
        "mauve_seed_start": int(MAUVE_STABILITY_BASE_SEED),
        "device_id": int(MAUVE_DEVICE_ID),
        "max_text_length": int(MAUVE_MAX_TEXT_LENGTH),
        "orientation": "p_text=real/reference; q_text=synthetic/generated",
        "interpretation": "stability/sensitivity diagnostic; not a formal confidence interval",
    }
    return frame, summary


if RUN_REPEATED_MAUVE:
    real_list = comparison_df["reference"].astype(str).tolist()
    synth_list = comparison_df["source"].astype(str).tolist()
    n_balanced = min(len(real_list), len(synth_list))

    stability_frames = []
    stability_summaries = []

    full_frame, full_summary = repeated_mauve(
        real_texts=real_list,
        synth_texts=synth_list,
        n_runs=REPEATED_MAUVE_N_RUNS,
        sample_size=n_balanced,
        sampling_seed=MAUVE_STABILITY_BASE_SEED,
        label="full_corpus_seed_stability",
    )
    stability_frames.append(full_frame)
    stability_summaries.append(full_summary)

    if MAUVE_SUBSAMPLE_FRACTION is not None:
        if not 0 < MAUVE_SUBSAMPLE_FRACTION < 1:
            raise ValueError("MAUVE_SUBSAMPLE_FRACTION must be between 0 and 1, or None.")
        subsample_size = max(2, int(np.floor(n_balanced * MAUVE_SUBSAMPLE_FRACTION)))
        subsample_frame, subsample_summary = repeated_mauve(
            real_texts=real_list,
            synth_texts=synth_list,
            n_runs=REPEATED_MAUVE_N_RUNS,
            sample_size=subsample_size,
            sampling_seed=MAUVE_STABILITY_BASE_SEED,
            label="subsample_sensitivity",
        )
        subsample_summary["subsample_fraction"] = float(MAUVE_SUBSAMPLE_FRACTION)
        stability_frames.append(subsample_frame)
        stability_summaries.append(subsample_summary)

    repeated_mauve_frame = pd.concat(stability_frames, ignore_index=True)
    repeated_mauve_frame.to_csv(RESULTS_DIR / "repeated_mauve_scores.csv", index=False)
    (RESULTS_DIR / "repeated_mauve_summary.json").write_text(
        json.dumps(stability_summaries, indent=2) + "\n"
    )
    display(repeated_mauve_frame)
    display(
        repeated_mauve_frame.groupby("analysis")["mauve"]
        .agg(["count", "mean", "std", "min", "max"]).reset_index()
    )
else:
    print("Repeated MAUVE skipped because RUN_REPEATED_MAUVE is False.")


## Save results to Drive, create a verified ZIP, and download locally

The export is **compact by default**: trained BERT `model/` and trainer/tester directories are omitted because they can make the archive several gigabytes. Set `EXPORT_INCLUDE_MODEL_CHECKPOINTS = True` in the configuration cell if checkpoint weights must be archived too.

The first cell below writes a run/environment manifest, copies the result bundle to Google Drive, creates a ZIP, checks the ZIP integrity and SHA-256 checksum, copies the ZIP to Drive, and starts the browser download. The browser's local Downloads folder is not visible to the Colab Python runtime, so local completion cannot be programmatically verified from the server.

**Only after the ZIP is visibly present on your computer should you run the final cell.** It asks for an explicit confirmation string before calling `runtime.unassign()`.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import zipfile

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

export_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
export_name = f"{EXPORT_ARCHIVE_PREFIX}_{export_timestamp}"
DRIVE_RUN_DIR = DRIVE_EXPORT_ROOT / export_name
LOCAL_EXPORT_DIR = Path("/content") / export_name
LOCAL_ZIP_PATH = Path("/content") / f"{export_name}.zip"
DRIVE_ZIP_PATH = DRIVE_EXPORT_ROOT / LOCAL_ZIP_PATH.name

def git_head(repo_dir):
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
        ).strip()
    except Exception:
        return None

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def sha256_path(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

run_manifest = {
    "export_timestamp_utc": export_timestamp,
    "project_root": str(PROJECT_ROOT),
    "results_dir": str(RESULTS_DIR),
    "icl_pilot_git_head": git_head(PROJECT_ROOT),
    "synthtexteval_git_head": git_head(SYNTHTEXTEVAL_DIR),
    "synthtexteval_requested_revision": SYNTHTEXTEVAL_REVISION,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        name: package_version(name)
        for name in [
            "SynthTextEval", "transformers", "torch", "datasets", "mauve-text",
            "spacy", "scikit-learn", "pandas", "numpy"
        ]
    },
    "configuration": {
        "comparison_cohorts": list(COMPARISON_COHORTS),
        "comparison_groups": list(COMPARISON_GROUPS),
        "sample_seed": SAMPLE_SEED,
        "max_samples_per_group": MAX_SAMPLES_PER_GROUP,
        "classification_rounds": list(CLASSIFICATION_ROUNDS),
        "classifier_model": CLASSIFIER_MODEL,
        "label_mapping": LABEL_MAPPING,
        "run_classification": RUN_CLASSIFICATION,
        "run_gillam_external_test": RUN_GILLAM_EXTERNAL_TEST,
        "run_privacy_analysis": RUN_PRIVACY_ANALYSIS,
        "run_mauve": RUN_MAUVE,
        "run_repeated_mauve": RUN_REPEATED_MAUVE,
        "repeated_mauve_n_runs": REPEATED_MAUVE_N_RUNS,
        "mauve_subsample_fraction": MAUVE_SUBSAMPLE_FRACTION,
        "export_include_model_checkpoints": EXPORT_INCLUDE_MODEL_CHECKPOINTS,
    },
    "input_files": {
        str(TEXT_RECORDS_CSV): {
            "sha256": sha256_path(TEXT_RECORDS_CSV),
            "size_bytes": TEXT_RECORDS_CSV.stat().st_size,
        },
        str(GILLAM_EXTERNAL_TEXT_CSV): {
            "sha256": sha256_path(GILLAM_EXTERNAL_TEXT_CSV),
            "size_bytes": GILLAM_EXTERNAL_TEXT_CSV.stat().st_size,
        },
    },
}
(RESULTS_DIR / "run_environment_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, default=str) + "\n"
)

# Build a clean local export tree. The original RESULTS_DIR remains untouched.
if LOCAL_EXPORT_DIR.exists():
    shutil.rmtree(LOCAL_EXPORT_DIR)

excluded_dir_names = {"__pycache__"}
if not EXPORT_INCLUDE_MODEL_CHECKPOINTS:
    excluded_dir_names.update({"model", "trainer", "tester", "external_gillam_tester"})

def export_ignore(directory, names):
    ignored = []
    for name in names:
        path = Path(directory) / name
        if path.is_dir() and name in excluded_dir_names:
            ignored.append(name)
    return ignored

shutil.copytree(RESULTS_DIR, LOCAL_EXPORT_DIR / "results", ignore=export_ignore)

# Add a checksum manifest for every portable result file.
exported_files = []
for path in sorted((LOCAL_EXPORT_DIR / "results").rglob("*")):
    if path.is_file():
        exported_files.append({
            "path": str(path.relative_to(LOCAL_EXPORT_DIR)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_path(path),
        })
(LOCAL_EXPORT_DIR / "portable_export_file_manifest.json").write_text(
    json.dumps(exported_files, indent=2) + "\n"
)

# Copy the portable directory to Drive and verify it exists before zipping.
if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(LOCAL_EXPORT_DIR, DRIVE_RUN_DIR)
assert (DRIVE_RUN_DIR / "results" / "run_environment_manifest.json").exists()

# Create the local ZIP from the portable export directory.
if LOCAL_ZIP_PATH.exists():
    LOCAL_ZIP_PATH.unlink()
shutil.make_archive(
    str(LOCAL_ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=LOCAL_EXPORT_DIR.parent,
    base_dir=LOCAL_EXPORT_DIR.name,
)
assert LOCAL_ZIP_PATH.exists() and LOCAL_ZIP_PATH.stat().st_size > 0

with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    assert bad_member is None, f"ZIP integrity check failed at: {bad_member}"
    zip_member_count = len(archive.infolist())

local_zip_sha256 = sha256_path(LOCAL_ZIP_PATH)
shutil.copy2(LOCAL_ZIP_PATH, DRIVE_ZIP_PATH)
assert DRIVE_ZIP_PATH.exists()
drive_zip_sha256 = sha256_path(DRIVE_ZIP_PATH)
assert local_zip_sha256 == drive_zip_sha256, "Drive ZIP checksum differs from local ZIP."

download_state = {
    "drive_result_directory": str(DRIVE_RUN_DIR),
    "drive_zip": str(DRIVE_ZIP_PATH),
    "local_zip_for_browser_download": str(LOCAL_ZIP_PATH),
    "zip_size_bytes": int(LOCAL_ZIP_PATH.stat().st_size),
    "zip_members": int(zip_member_count),
    "zip_sha256": local_zip_sha256,
    "server_side_checks_passed": True,
    "browser_download_confirmed": False,
}
(DRIVE_RUN_DIR / "download_state_before_local_confirmation.json").write_text(
    json.dumps(download_state, indent=2) + "\n"
)

print(json.dumps(download_state, indent=2))
print("\nServer-side Drive copy, ZIP integrity, and checksum checks passed.")
print("Starting browser download now. Do NOT run the final cell until the ZIP is visibly present on your computer.")
files.download(str(LOCAL_ZIP_PATH))

### Final step: confirm the local download, then release the Colab runtime

Run this cell **manually after checking your computer's Downloads folder**. Typing anything other than the exact confirmation string leaves the runtime assigned.

In [ ]:
# This MUST remain the final executable cell in the notebook.
confirmation = input(
    f"Confirm that {LOCAL_ZIP_PATH.name} is fully downloaded on your computer. "
    "Type exactly DOWNLOADED to unassign this Colab runtime: "
).strip()

if confirmation != "DOWNLOADED":
    raise RuntimeError(
        "Local download was not explicitly confirmed. Runtime remains assigned. "
        "Verify the ZIP in your Downloads folder, then rerun this final cell."
    )

# Recheck the durable Drive copy immediately before releasing compute.
assert DRIVE_ZIP_PATH.exists(), f"Drive ZIP is missing: {DRIVE_ZIP_PATH}"
assert sha256_path(DRIVE_ZIP_PATH) == local_zip_sha256, "Drive ZIP checksum changed."

confirmed_state = dict(download_state)
confirmed_state["browser_download_confirmed"] = True
confirmed_state["confirmation_method"] = "manual user verification in browser/local Downloads folder"
(DRIVE_RUN_DIR / "download_state_confirmed.json").write_text(
    json.dumps(confirmed_state, indent=2) + "\n"
)
print("Local download confirmed and Drive archive rechecked. Unassigning Colab runtime.")

from google.colab import runtime
runtime.unassign()